# F1 Lakehouse — Exploration Notebook

A scratchpad for poking at the pipeline while you build it.

**What's in here:**
1. Setup — imports + project paths
2. Inspect the `raw_races` Delta table that Dagster wrote (now **partitioned by season**)
3. Explore the source API directly (Jolpica)
4. Query Delta from DuckDB (same engine dbt uses)
5. Delta time travel — read past versions
6. Silver layer — staging models (races, drivers, results, sprint results)
7. Gold layer — the marts (per-season standings; `total_points` = GP + sprint)
8. Ad-hoc analysis — Polars + DuckDB sandbox

**Seasons:** the bronze layer ingests **2023 + 2024** as Dagster partitions, stored in one
Delta table partitioned by a `season` column. The marts are grained per season and fold
sprint points into the championship totals.

**Run it:** `uv run jupyter lab` from project root.

## Setup

In [1]:
from pathlib import Path

import duckdb
import httpx
import polars as pl
from deltalake import DeltaTable

# Resolve project root whether notebook is launched from root or from notebooks/
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data"

# The lakehouse is namespaced by dataset (the stack/dataset split — see CLAUDE.md).
# Every zone lives under data/<zone>/<dataset>/, so paths are built from DATASET.
# Point this at another dataset (e.g. "nyc_taxi") to explore it with the same cells.
DATASET = "f1"

# Bronze (Delta — PolarsDeltaIOManager appends '.delta' as a format marker).
# Each table is partitioned by `season` (2023, 2024) inside the one .delta directory.
RAW_RACES_PATH          = DATA_ROOT / "raw" / DATASET / "raw_races.delta"
RAW_DRIVERS_PATH        = DATA_ROOT / "raw" / DATASET / "raw_drivers.delta"
RAW_RESULTS_PATH        = DATA_ROOT / "raw" / DATASET / "raw_results.delta"
RAW_SPRINT_RESULTS_PATH = DATA_ROOT / "raw" / DATASET / "raw_sprint_results.delta"

# Silver (Parquet — dbt external materialization)
STG_RACES_PATH          = DATA_ROOT / "staging" / DATASET / "stg_races.parquet"
STG_DRIVERS_PATH        = DATA_ROOT / "staging" / DATASET / "stg_drivers.parquet"
STG_RESULTS_PATH        = DATA_ROOT / "staging" / DATASET / "stg_results.parquet"
STG_SPRINT_RESULTS_PATH = DATA_ROOT / "staging" / DATASET / "stg_sprint_results.parquet"

# Gold (Parquet — dbt external materialization)
MART_COUNTRIES_PATH   = DATA_ROOT / "marts" / DATASET / "mart_country_race_summary.parquet"
MART_STANDINGS_PATH   = DATA_ROOT / "marts" / DATASET / "mart_driver_standings.parquet"
MART_CONSTRUCTOR_PATH = DATA_ROOT / "marts" / DATASET / "mart_constructor_standings.parquet"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Dataset:      {DATASET}")
print(f"Polars:       {pl.__version__}")
print(f"DuckDB:       {duckdb.__version__}")

Project root: /Users/abhisheksingh/Documents/study/polars-dbt-duckdb-dagster
Data root:    /Users/abhisheksingh/Documents/study/polars-dbt-duckdb-dagster/data
Dataset:      f1
Polars:       1.41.2
DuckDB:       1.5.3


## 1. Inspect the raw Delta table

Read `data/raw/f1/raw_races.delta/` — the Delta table Polars wrote during the last Dagster materialization (namespaced under the `f1/` dataset). It now holds **both seasons**, partitioned by a `season` column (`season=2023/`, `season=2024/` directories on disk).

In [2]:
# Read the whole Delta table into a Polars DataFrame
df = pl.read_delta(str(RAW_RACES_PATH))
df.head()

round,url,raceName,Circuit,date,time,FirstPractice,SecondPractice,ThirdPractice,Qualifying,Sprint,SprintShootout,season,SprintQualifying
str,str,str,struct[4],str,str,struct[2],struct[2],struct[2],struct[2],struct[2],struct[2],str,struct[2]
"""1""","""https://en.wikipedia.org/wiki/…","""Bahrain Grand Prix""","{""bahrain"",""https://en.wikipedia.org/wiki/Bahrain_International_Circuit"",""Bahrain International Circuit"",{""26.0325"",""50.5106"",""Sakhir"",""Bahrain""}}","""2023-03-05""","""15:00:00Z""","{""2023-03-03"",""11:30:00Z""}","{""2023-03-03"",""15:00:00Z""}","{""2023-03-04"",""11:30:00Z""}","{""2023-03-04"",""15:00:00Z""}",null,null,"""2023""",null
"""2""","""https://en.wikipedia.org/wiki/…","""Saudi Arabian Grand Prix""","{""jeddah"",""https://en.wikipedia.org/wiki/Jeddah_Corniche_Circuit"",""Jeddah Corniche Circuit"",{""21.6319"",""39.1044"",""Jeddah"",""Saudi Arabia""}}","""2023-03-19""","""17:00:00Z""","{""2023-03-17"",""13:30:00Z""}","{""2023-03-17"",""17:00:00Z""}","{""2023-03-18"",""13:30:00Z""}","{""2023-03-18"",""17:00:00Z""}",null,null,"""2023""",null
"""3""","""https://en.wikipedia.org/wiki/…","""Australian Grand Prix""","{""albert_park"",""https://en.wikipedia.org/wiki/Albert_Park_Circuit"",""Albert Park Grand Prix Circuit"",{""-37.8497"",""144.968"",""Melbourne"",""Australia""}}","""2023-04-02""","""05:00:00Z""","{""2023-03-31"",""01:30:00Z""}","{""2023-03-31"",""05:00:00Z""}","{""2023-04-01"",""01:30:00Z""}","{""2023-04-01"",""05:00:00Z""}",null,null,"""2023""",null
"""4""","""https://en.wikipedia.org/wiki/…","""Azerbaijan Grand Prix""","{""baku"",""https://en.wikipedia.org/wiki/Baku_City_Circuit"",""Baku City Circuit"",{""40.3725"",""49.8533"",""Baku"",""Azerbaijan""}}","""2023-04-30""","""11:00:00Z""","{""2023-04-28"",""09:30:00Z""}",null,null,"{""2023-04-28"",""13:00:00Z""}","{""2023-04-29"",""13:30:00Z""}","{""2023-04-29"",""09:30:00Z""}","""2023""",null
"""5""","""https://en.wikipedia.org/wiki/…","""Miami Grand Prix""","{""miami"",""https://en.wikipedia.org/wiki/Miami_International_Autodrome"",""Miami International Autodrome"",{""25.9581"",""-80.2389"",""Miami"",""USA""}}","""2023-05-07""","""19:30:00Z""","{""2023-05-05"",""18:00:00Z""}","{""2023-05-05"",""21:30:00Z""}","{""2023-05-06"",""16:30:00Z""}","{""2023-05-06"",""20:00:00Z""}",null,null,"""2023""",null


In [3]:
# Schema (column names + Polars dtypes). Nested objects from the API show up as Struct columns.
df.schema

Schema([('round', String),
        ('url', String),
        ('raceName', String),
        ('Circuit',
         Struct({'circuitId': String, 'url': String, 'circuitName': String, 'Location': Struct({'lat': String, 'long': String, 'locality': String, 'country': String})})),
        ('date', String),
        ('time', String),
        ('FirstPractice', Struct({'date': String, 'time': String})),
        ('SecondPractice', Struct({'date': String, 'time': String})),
        ('ThirdPractice', Struct({'date': String, 'time': String})),
        ('Qualifying', Struct({'date': String, 'time': String})),
        ('Sprint', Struct({'date': String, 'time': String})),
        ('SprintShootout', Struct({'date': String, 'time': String})),
        ('season', String),
        ('SprintQualifying', Struct({'date': String, 'time': String}))])

In [4]:
# Row count + null counts per column — useful before designing staging models
print(f"Rows: {df.height}")
df.null_count()

Rows: 46


round,url,raceName,Circuit,date,time,FirstPractice,SecondPractice,ThirdPractice,Qualifying,Sprint,SprintShootout,season,SprintQualifying
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,12,12,0,34,40,0,40


In [5]:
# Partitioning check: rows per season in the single Delta table.
# Re-materializing one season replaces only its slice (predicate overwrite) — the other survives.
df.group_by("season").len().sort("season")

season,len
str,u32
"""2023""",22
"""2024""",24


## 2. Explore the source API

Sometimes the fastest way to understand a source is to just `GET` it and stare at one record.

In [6]:
# Pull one season of races straight from Jolpica
response = httpx.get("https://api.jolpi.ca/ergast/f1/2024/races.json", timeout=30.0)
response.raise_for_status()

payload = response.json()["MRData"]
print(f"Total races reported: {payload['total']}")
print(f"Top-level keys:       {list(payload['RaceTable']['Races'][0].keys())}")

Total races reported: 24
Top-level keys:       ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'FirstPractice', 'SecondPractice', 'ThirdPractice', 'Qualifying']


In [7]:
# Pretty-print one raw race record to understand the JSON shape
import json
print(json.dumps(payload["RaceTable"]["Races"][0], indent=2))

{
  "season": "2024",
  "round": "1",
  "url": "https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix",
  "raceName": "Bahrain Grand Prix",
  "Circuit": {
    "circuitId": "bahrain",
    "url": "https://en.wikipedia.org/wiki/Bahrain_International_Circuit",
    "circuitName": "Bahrain International Circuit",
    "Location": {
      "lat": "26.0325",
      "long": "50.5106",
      "locality": "Sakhir",
      "country": "Bahrain"
    }
  },
  "date": "2024-03-02",
  "time": "15:00:00Z",
  "FirstPractice": {
    "date": "2024-02-29",
    "time": "11:30:00Z"
  },
  "SecondPractice": {
    "date": "2024-02-29",
    "time": "15:00:00Z"
  },
  "ThirdPractice": {
    "date": "2024-03-01",
    "time": "12:30:00Z"
  },
  "Qualifying": {
    "date": "2024-03-01",
    "time": "16:00:00Z"
  }
}


## 3. Query the Delta table from DuckDB

DuckDB reads Delta tables natively via its `delta` extension. This is what dbt-duckdb uses under the hood — useful for prototyping a staging query before committing it to a dbt model.

In [8]:
conn = duckdb.connect()
conn.execute("INSTALL delta; LOAD delta;")

# Identifier-quoting note: Ergast JSON uses camelCase keys (raceName, etc.),
# so we keep them quoted in SQL. dbt staging models will rename to snake_case.
conn.execute(f"""
    SELECT season, round, "raceName" AS race_name, "date" AS race_date
    FROM delta_scan('{RAW_RACES_PATH}')
    ORDER BY CAST(season AS INTEGER), CAST(round AS INTEGER)
    LIMIT 10
""").pl()  # .pl() returns a Polars DataFrame

season,round,race_name,race_date
str,str,str,str
"""2023""","""1""","""Bahrain Grand Prix""","""2023-03-05"""
"""2023""","""2""","""Saudi Arabian Grand Prix""","""2023-03-19"""
"""2023""","""3""","""Australian Grand Prix""","""2023-04-02"""
"""2023""","""4""","""Azerbaijan Grand Prix""","""2023-04-30"""
"""2023""","""5""","""Miami Grand Prix""","""2023-05-07"""
"""2023""","""6""","""Monaco Grand Prix""","""2023-05-28"""
"""2023""","""7""","""Spanish Grand Prix""","""2023-06-04"""
"""2023""","""8""","""Canadian Grand Prix""","""2023-06-18"""
"""2023""","""9""","""Austrian Grand Prix""","""2023-07-02"""


## 4. Delta time travel

Every write to a Delta table creates a new version. You can read any past version — useful when debugging "why is this row different from yesterday?".

In [9]:
dt = DeltaTable(str(RAW_RACES_PATH))

print(f"Current version: {dt.version()}")
print("\nHistory (oldest first):")
for entry in dt.history():
    print(f"  v{entry['version']}: {entry.get('operation', '?')} at {entry.get('timestamp', '?')}")

Current version: 2

History (oldest first):
  v2: WRITE at 1782195697347
  v1: WRITE at 1782195598966
  v0: WRITE at 1782195567625


In [10]:
# Read a specific past version (only meaningful after multiple materializations)
# pl.read_delta(str(RAW_RACES_PATH), version=0)

## 5. Silver layer — what staging produced

Four staging models clean and conform the raw layer: `stg_races`, `stg_drivers`,
`stg_results`, and `stg_sprint_results`. Each is a single Parquet file and carries a
`season` column so downstream marts can group by it.

> Run `Materialize` on the staging assets (or any downstream mart) in Dagster first — these cells will error if the files don't exist yet.

In [11]:
# Drivers, one row per (driver, season) — typed dates, snake_case names, convenience full_name.
# A driver who raced both seasons appears once per season (joins downstream use driver_id + season).
stg_drivers = pl.read_parquet(STG_DRIVERS_PATH)
print(f"Rows: {stg_drivers.height}")
stg_drivers.head()

Rows: 47


season,driver_id,permanent_number,driver_code,given_name,family_name,full_name,date_of_birth,nationality,driver_url
i32,str,i32,str,str,str,str,date,str,str
2024,"""albon""",23,"""ALB""","""Alexander""","""Albon""","""Alexander Albon""",1996-03-23,"""Thai""","""http://en.wikipedia.org/wiki/A…"
2024,"""alonso""",14,"""ALO""","""Fernando""","""Alonso""","""Fernando Alonso""",1981-07-29,"""Spanish""","""http://en.wikipedia.org/wiki/F…"
2024,"""antonelli""",12,"""ANT""","""Andrea Kimi""","""Antonelli""","""Andrea Kimi Antonelli""",2006-08-25,"""Italian""","""https://en.wikipedia.org/wiki/…"
2024,"""bearman""",87,"""BEA""","""Oliver""","""Bearman""","""Oliver Bearman""",2005-05-08,"""British""","""http://en.wikipedia.org/wiki/O…"
2024,"""bottas""",77,"""BOT""","""Valtteri""","""Bottas""","""Valtteri Bottas""",1989-08-28,"""Finnish""","""http://en.wikipedia.org/wiki/V…"


In [12]:
# Grand Prix results: one row per (race, driver) across both seasons
stg_results = pl.read_parquet(STG_RESULTS_PATH)
print(f"Rows: {stg_results.height}")
stg_results.head()

Rows: 919


season,round,race_name,driver_id,constructor_id,constructor_name,car_number,position_raw,finishing_position,points,grid_position,laps_completed,finish_status,race_time_ms,fastest_lap_rank,fastest_lap_number
i32,i32,str,str,str,str,i32,str,i32,f64,i32,i32,str,i64,i32,i32
2024,1,"""Bahrain Grand Prix""","""max_verstappen""","""red_bull""","""Red Bull""",1,"""1""",1,26.0,1,57,"""Finished""",5504742,1,39
2024,1,"""Bahrain Grand Prix""","""perez""","""red_bull""","""Red Bull""",11,"""2""",2,18.0,5,57,"""Finished""",5527199,4,40
2024,1,"""Bahrain Grand Prix""","""sainz""","""ferrari""","""Ferrari""",55,"""3""",3,15.0,4,57,"""Finished""",5529852,6,44
2024,1,"""Bahrain Grand Prix""","""leclerc""","""ferrari""","""Ferrari""",16,"""4""",4,12.0,2,57,"""Finished""",5544411,2,36
2024,1,"""Bahrain Grand Prix""","""russell""","""mercedes""","""Mercedes""",63,"""5""",5,10.0,3,57,"""Finished""",5551530,12,40


In [13]:
stg_results.select(pl.col("season").unique().sort()).to_series().to_list()

[2023, 2024]

In [14]:
# Sprint results: one row per (sprint, driver). Only the ~6 sprint weekends/season have these.
# Their points are added on top of GP points in the standings marts.
stg_sprint = pl.read_parquet(STG_SPRINT_RESULTS_PATH)
print(f"Rows: {stg_sprint.height}")
print("Sprint points per season:")
print(stg_sprint.group_by("season").agg(pl.col("points").sum().alias("sprint_points_awarded")).sort("season"))
stg_sprint.head()

Rows: 240
Sprint points per season:
shape: (2, 2)
┌────────┬───────────────────────┐
│ season ┆ sprint_points_awarded │
│ ---    ┆ ---                   │
│ i32    ┆ f64                   │
╞════════╪═══════════════════════╡
│ 2023   ┆ 216.0                 │
│ 2024   ┆ 216.0                 │
└────────┴───────────────────────┘


season,round,race_name,driver_id,constructor_id,constructor_name,position_raw,finishing_position,points,grid_position,finish_status
i32,i32,str,str,str,str,str,i32,f64,i32,str
2024,5,"""Chinese Grand Prix""","""max_verstappen""","""red_bull""","""Red Bull""","""1""",1,8.0,4,"""Finished"""
2024,5,"""Chinese Grand Prix""","""hamilton""","""mercedes""","""Mercedes""","""2""",2,7.0,2,"""Finished"""
2024,5,"""Chinese Grand Prix""","""perez""","""red_bull""","""Red Bull""","""3""",3,6.0,6,"""Finished"""
2024,5,"""Chinese Grand Prix""","""leclerc""","""ferrari""","""Ferrari""","""4""",4,5.0,7,"""Finished"""
2024,5,"""Chinese Grand Prix""","""sainz""","""ferrari""","""Ferrari""","""5""",5,4.0,5,"""Finished"""


## 6. Gold layer — the marts

Three marts answer business questions, all grained **per season**:
- `mart_country_race_summary` — one row per (country, season)
- `mart_driver_standings` — one row per (driver, season); `total_points` = `race_points` + `sprint_points`
- `mart_constructor_standings` — one row per (constructor, season); same points breakdown

Wins / podiums / pole_positions stay Grand-Prix-only; sprint contributes points only.

In [15]:
# mart_constructor_standings — championship summary per (constructor, season), 2024 shown.
constructor_standings = pl.read_parquet(MART_CONSTRUCTOR_PATH)
print(f"Rows (all seasons): {constructor_standings.height}")
ccols = ["season", "constructor_name", "race_points", "sprint_points", "total_points",
         "wins", "podiums", "pole_positions"]
with pl.Config(tbl_cols=-1, tbl_width_chars=200):
    print(constructor_standings.filter(pl.col("season") == 2024).select(ccols).sort("total_points", descending=True))

Rows (all seasons): 20
shape: (10, 8)
┌────────┬──────────────────┬─────────────┬───────────────┬──────────────┬──────┬─────────┬────────────────┐
│ season ┆ constructor_name ┆ race_points ┆ sprint_points ┆ total_points ┆ wins ┆ podiums ┆ pole_positions │
│ ---    ┆ ---              ┆ ---         ┆ ---           ┆ ---          ┆ ---  ┆ ---     ┆ ---            │
│ i32    ┆ str              ┆ f64         ┆ f64           ┆ f64          ┆ f64  ┆ f64     ┆ f64            │
╞════════╪══════════════════╪═════════════╪═══════════════╪══════════════╪══════╪═════════╪════════════════╡
│ 2024   ┆ McLaren          ┆ 609.0       ┆ 57.0          ┆ 666.0        ┆ 6.0  ┆ 21.0    ┆ 8.0            │
│ 2024   ┆ Ferrari          ┆ 595.0       ┆ 57.0          ┆ 652.0        ┆ 5.0  ┆ 22.0    ┆ 4.0            │
│ 2024   ┆ Red Bull         ┆ 537.0       ┆ 52.0          ┆ 589.0        ┆ 9.0  ┆ 18.0    ┆ 8.0            │
│ 2024   ┆ Mercedes         ┆ 433.0       ┆ 35.0          ┆ 468.0        ┆ 4.0  ┆ 9.0     

In [16]:
# mart_driver_standings — championship summary per (driver, season).
# total_points = race_points + sprint_points. Showing the 2024 top of the table.
standings = pl.read_parquet(MART_STANDINGS_PATH)
print(f"Rows (all seasons): {standings.height}")
cols = ["season", "driver_name", "primary_constructor", "race_points", "sprint_points",
        "total_points", "wins", "podiums", "pole_positions"]
with pl.Config(tbl_cols=-1, tbl_width_chars=200):
    print(standings.filter(pl.col("season") == 2024).select(cols).sort("total_points", descending=True).head(10))

Rows (all seasons): 46
shape: (10, 9)
┌────────┬─────────────────┬─────────────────────┬─────────────┬───────────────┬──────────────┬──────┬─────────┬────────────────┐
│ season ┆ driver_name     ┆ primary_constructor ┆ race_points ┆ sprint_points ┆ total_points ┆ wins ┆ podiums ┆ pole_positions │
│ ---    ┆ ---             ┆ ---                 ┆ ---         ┆ ---           ┆ ---          ┆ ---  ┆ ---     ┆ ---            │
│ i32    ┆ str             ┆ str                 ┆ f64         ┆ f64           ┆ f64          ┆ f64  ┆ f64     ┆ f64            │
╞════════╪═════════════════╪═════════════════════╪═════════════╪═══════════════╪══════════════╪══════╪═════════╪════════════════╡
│ 2024   ┆ Max Verstappen  ┆ Red Bull            ┆ 399.0       ┆ 38.0          ┆ 437.0        ┆ 9.0  ┆ 14.0    ┆ 8.0            │
│ 2024   ┆ Lando Norris    ┆ McLaren             ┆ 344.0       ┆ 30.0          ┆ 374.0        ┆ 4.0  ┆ 13.0    ┆ 8.0            │
│ 2024   ┆ Charles Leclerc ┆ Ferrari             ┆ 3

In [17]:
# mart_constructor_standings — championship-style summary, sorted by points
constructor_standings = pl.read_parquet(MART_CONSTRUCTOR_PATH)
with pl.Config(tbl_cols=-1, tbl_width_chars=200):
    print(constructor_standings)

shape: (20, 13)
┌────────────────────────┬────────┬──────────────────┬──────────────┬─────────────┬───────────────┬───────────────┬──────┬─────────┬─────────────────┬──────┬────────────────┬────────────────────────┐
│ constructor_season_key ┆ season ┆ constructor_name ┆ total_points ┆ race_points ┆ sprint_points ┆ races_entered ┆ wins ┆ podiums ┆ points_finishes ┆ dnfs ┆ pole_positions ┆ avg_finishing_position │
│ ---                    ┆ ---    ┆ ---              ┆ ---          ┆ ---         ┆ ---           ┆ ---           ┆ ---  ┆ ---     ┆ ---             ┆ ---  ┆ ---            ┆ ---                    │
│ str                    ┆ i32    ┆ str              ┆ f64          ┆ f64         ┆ f64           ┆ i64           ┆ f64  ┆ f64     ┆ f64             ┆ f64  ┆ f64            ┆ f64                    │
╞════════════════════════╪════════╪══════════════════╪══════════════╪═════════════╪═══════════════╪═══════════════╪══════╪═════════╪═════════════════╪══════╪════════════════╪══════════

## 7. Ad-hoc analysis — Polars + DuckDB sandbox

You don't have to go through dbt to explore the data. Polars joins across the staging Parquet files directly, and DuckDB can SQL them. Use this for quick experiments before deciding what becomes a permanent dbt model.

In [18]:
# Example: total championship points (GP + sprint) by driver nationality, for 2024.
# Pure Polars — joins stg_results + stg_drivers WITHOUT the marts.
# Join on (driver_id, season): drivers are per-season now, so an unqualified join fans out.
gp_2024 = stg_results.filter(pl.col("season") == 2024)
sprint_2024 = stg_sprint.filter(pl.col("season") == 2024)

points_by_nation = (
    pl.concat([
        gp_2024.select("driver_id", "season", "points"),
        sprint_2024.select("driver_id", "season", "points"),
    ])
    .join(stg_drivers, on=["driver_id", "season"])
    .group_by("nationality")
    .agg(pl.col("points").sum().alias("total_points"))
    .sort("total_points", descending=True)
)
points_by_nation.head(10)

nationality,total_points
str,f64
"""British""",849.0
"""Dutch""",437.0
"""Spanish""",360.0
"""Monegasque""",356.0
"""Australian""",304.0
"""Mexican""",152.0
"""French""",65.0
"""German""",41.0
"""Japanese""",30.0
